# BAB 4: GROUPING & AGGREGATION

Grouping & Aggregation adalah teknik untuk memecah data berdasarkan kelompok kategori tertentu (*split*), menerapkan fungsi perhitungan statistik pada kelompok tersebut (*apply*), lalu menggabungkan kembali hasilnya ke dalam tabel ringkasan (*combine*). Proses ini sangat vital dalam tahap *Data Exploration* dan *Feature Engineering* untuk mengenali pola tersembunyi antar kelompok data.

---

## A. Fondasi Utama Groupby & Agregasi Langsung
Metode dasar yang digunakan untuk menghitung satu nilai statistik pada satu kolom target numerik setelah dikelompokkan berdasarkan kolom kategori.

1. **Single-Level Groupby**
   * **Konsep:** Mengelompokkan data berdasarkan satu kolom kategori, lalu menghitung satu fungsi statistik (seperti `.mean()`, `.sum()`, `.max()`, `.min()`, atau `.count()`) pada kolom numerik yang ditargetkan.
   * **Formula Pandas:** `df.groupby('Kolom_Kategori')['Kolom_Numerik'].fungsi_statistik()`
   * **Contoh:** Menghitung rata-rata pendapatan berdasarkan jenis kelamin.

2. **Multi-Level Groupby**
   * **Konsep:** Mengelompokkan data berdasarkan kombinasi dua kategori atau lebih sekaligus secara berjenjang. Kolom pengelompok dimasukkan ke dalam bentuk *list* Python `['Kolom_A', 'Kolom_B']`.
   * **Formula Pandas:** `df.groupby(['Kategori_1', 'Kategori_2'])['Kolom_Numerik'].fungsi_statistik()`
   * **Contoh:** Menghitung rata-rata usia pasien berdasarkan kombinasi `jenis_kelamin` dan `status_merokok`.

---

## B. Agregasi Tingkat Lanjut via `.agg()`
Ketika metode dasar hanya mampu menghitung satu fungsi statistik saja, fungsi `.agg()` hadir sebagai senjata utama untuk melakukan banyak kalkulasi sekaligus dalam satu baris perintah.

1. **Agregasi Berbasis List (Multi-Fungsi Seragam)**
   * **Konsep:** Menerapkan beberapa fungsi statistik sekaligus ke seluruh kolom numerik yang dipilih.
   * **Formula Pandas:** `df.groupby('Kategori')[['Kolom_Num1', 'Kolom_Num2']].agg(['min', 'max', 'mean'])`
   * **Karakteristik:** Menghasilkan kolom bertingkat (*Multi-Index Column*) yang memuat nilai minimum, maksimum, dan rata-rata untuk setiap kolom numerik yang didaftarkan.

2. **Agregasi Berbasis Dictionary (Kustom Per Kolom)**
   * **Konsep:** Pendekatan paling fleksibel di industri di mana kita bisa memetakan secara spesifik fungsi statistik yang berbeda untuk setiap kolom numerik menggunakan *dictionary* Python `{ 'nama_kolom': 'fungsi' }`.
   * **Formula Pandas:** 
     ```python
     perintah = {'usia': 'mean', 'pendapatan': 'sum'}
     df.groupby('Kategori').agg(perintah)
     ```

3. **Named Aggregation (Modern & Rapi)**
   * **Konsep:** Fitur terbaik Pandas untuk melakukan agregasi kustom sekaligus langsung mendeklarasikan nama kolom baru yang bersih (format *flat column*, bukan *multi-index* bertingkat) agar mudah diakses di tahap selanjutnya.
   * **Formula Pandas:**
     ```python
     df.groupby('jenis_kelamin').agg(
         total_pendapatan=('pendapatan', 'sum'),
         rata_usia=('usia', 'mean')
     )
     ```

---

## C. Anatomi & Aturan Penting Groupby (Jebakan Batman)
Karakteristik teknis bawaan Pandas Groupby yang wajib dipahami agar tidak memicu *error* atau kebingungan saat manipulasi data.

1. **Perubahan Status Indeks (`as_index=False`)**
   * **Karakteristik:** Secara default, kolom yang dijadikan dasar pengelompokan (misal `jenis_kelamin`) otomatis akan berubah status menjadi **Index** dari DataFrame hasil, bukan lagi menjadi kolom biasa.
   * **Solusi Penyelamat:** Jika ingin hasil agregasi tetap mempertahankan format DataFrame normal dengan kolom-kolom biasa yang sejajar, tambahkan parameter `as_index=False` di dalam fungsi groupby.
   * **Contoh:** `df.groupby('jenis_kelamin', as_index=False)['pendapatan'].mean()`

2. **Perbedaan Mutlak `count()` vs `size()`**
   * **Fungsi `.count()`:** Hanya menghitung jumlah baris data yang **memiliki isi (valid)** pada kelompok tersebut. Jika ada data yang kosong (*Missing Value / NaN*), baris tersebut akan dilewati (tidak dihitung).
   * **Fungsi `.size()`:** Menghitung **seluruh total baris fisik** yang ada di dalam kelompok tersebut secara mutlak, tanpa peduli apakah di dalamnya ada data yang kosong (*NaN*) atau tidak.

In [1]:
import pandas as pd

In [2]:
# dataset
data = {
    'jenis_kelamin': ['Pria', 'Wanita', 'Pria', 'Wanita', 'Pria'],
    'status_merokok': ['Sering', 'Tidak Pernah', 'Jarang', 'Sering', 'Tidak Pernah'],
    'usia': [45, 23, 34, 50, 29],
    'pendapatan': [15000000, 8000000, 12000000, 20000000, 9500000]
}
df = pd.DataFrame(data)

In [4]:
# Fondasi Utama (grouping)
# Kelompokkan berdasarkan jenis kelamin, lalu ambil rata-rata pendapatan dari masing-masing jenis kelamin
rata_pendapatan_jk = df.groupby('jenis_kelamin')['pendapatan'].mean()
display(rata_pendapatan_jk)

jenis_kelamin
Pria      1.216667e+07
Wanita    1.400000e+07
Name: pendapatan, dtype: float64

In [7]:
# Multi-Level Groupby
# Rata-rata usia pasien jika dikelompokkan berdasarkan jenis kelamin dan status merokok
rata_usia_kombinasi = df.groupby(["jenis_kelamin", "status_merokok"])["usia"].mean()
display(rata_usia_kombinasi)

jenis_kelamin  status_merokok
Pria           Jarang            34.0
               Sering            45.0
               Tidak Pernah      29.0
Wanita         Sering            50.0
               Tidak Pernah      23.0
Name: usia, dtype: float64

In [ ]:
# Multi-Function Groupby
# Dapat menghitung banyak hal sekaligus
perintah_agg = {
    'usia': 'mean',          # Kolom usia dihitung rata-ratanya
    'pendapatan': ['sum', 'max'], # Kolom pendapatan dihitung total & nilai tertingginya
    'status_merokok': 'count' # Kolom ini dihitung ada berapa baris data (jumlah pasien)
}

hasil_custom = df.groupby('jenis_kelamin').agg(perintah_agg)
display(hasil_custom)

usia pendapatan           status_merokok
               mean        sum       max          count
jenis_kelamin                                          
Pria           36.0   36500000  15000000              3
Wanita         36.5   28000000  20000000              2